In [1]:
from classes.stock_client import StockClient
from classes.quant_analytics import QuantAnalytics as qa
client = StockClient(ttl_price=60, ttl_info=900)
quant = qa(client)

In [2]:
from modules.factors import run, run_bulk, compare_models, rolling_betas

In [3]:
# single stock, single model
result = run(quant, "AAPL", model="ff5", period="5y")
print(result)

FactorResult — AAPL  [FF5]  5y
  Alpha (ann.)  : +2.756%   (p=0.731)
  R²            : 0.608   Adj-R²: 0.606   N: 1221

  Factor       Beta   t-stat    p-val
  --------------------------------------
  Mkt-RF      1.223    38.33    0.000 ***
  SMB        -0.070    -1.30    0.194 
  HML        -0.393    -8.05    0.000 ***
  RMW         0.514     8.60    0.000 ***
  CMA         0.268     3.93    0.000 ***


In [5]:
# all four models side by side
df = compare_models(quant, "AAPL", period="5y")
df

,alpha,alpha_pval,r_squared,adj_r2,n_obs
FF3,0.038183,0.646708,0.578455,0.577416,1221.0
FF5,0.027561,0.730594,0.607642,0.606027,1221.0
MOM,0.045672,0.582074,0.585339,0.583975,1221.0
FF6,0.036863,0.643670,0.615092,0.613190,1221.0


In [7]:
# entire basket at once
df = run_bulk(quant, ["AAPL","MSFT","NVDA"], model="ff5", period="3y")
df

,alpha,r_squared,Mkt-RF,SMB,HML,RMW,CMA
AAPL,-0.023057,0.482871,1.219330,-0.005185,-0.259867,0.702240,0.110771
MSFT,-0.085541,0.545516,0.954615,-0.237808,-0.399069,0.190850,-0.636693
NVDA,0.534343,0.540612,2.035116,-0.615799,-1.242940,0.442639,0.264247


In [8]:
# how exposures shift over time
df = rolling_betas(quant, "AAPL", model="ff3", window=126)
df

,alpha_daily,Mkt-RF,SMB,HML
2021-10-15,-0.000129,1.383102,-0.458037,-0.244796
2021-10-18,-0.000008,1.378034,-0.475173,-0.256052
2021-10-19,0.000051,1.385900,-0.466521,-0.256824
2021-10-20,0.000061,1.386932,-0.467284,-0.255189
2021-10-21,-0.000022,1.376507,-0.474304,-0.253583
...,...,...,...,...
2026-02-23,0.000685,0.969760,-0.366589,0.171644
2026-02-24,0.000821,0.977659,-0.355820,0.153119
2026-02-25,0.000818,0.971430,-0.352959,0.150325
2026-02-26,0.000787,0.967875,-0.352945,0.149109


Reading the output:

alpha > 0 and significant → the stock generated returns beyond what the factors explain
High Mkt-RF beta → amplifies market moves
Positive SMB → behaves like a small-cap even if it isn't
Positive HML → value tilt; negative → growth tilt
High R² → most of the stock's variance is explained by systematic factors; low R² → more idiosyncratic

In [10]:
import modules.plots as plots

In [11]:
plots.factor_exposure(result)

In [12]:
plots.rolling_factor_betas(quant, 'AAPL')